In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
def encodeLabel(data, feature):
    encoder = LabelEncoder()
    data[feature] = encoder.fit_transform(data[feature].fillna('none'))
    return data

In [3]:
driver = pd.read_csv('../data/driver.csv')

In [4]:
item = pd.read_csv('../data/data/item_data.csv')
item = encodeLabel(item, 'brand')
item = encodeLabel(item, 'brand_type')
item = encodeLabel(item, 'category')

In [5]:
data = pd.read_csv('../data/data/customer_transaction_data.csv')
data = data[data['coupon_discount'] == 0]
data = data.merge(item[['item_id','category']], on='item_id').drop('item_id', axis=1)
data = data.groupby(['customer_id','category'])[['selling_price','quantity']].sum().reset_index()

In [6]:
mapping = pd.read_csv('../data/data/coupon_item_mapping.csv')
mapping = mapping.merge(item, on='item_id')[['coupon_id','category']]
mapping = mapping.drop_duplicates()

In [7]:
data = data.merge(mapping, on=['category'])

In [8]:
data.head()

,customer_id,category,selling_price,quantity,coupon_id
0,1,1,2826.38,39,106
1,1,1,2826.38,39,958
2,1,1,2826.38,39,164
3,1,1,2826.38,39,122
4,1,1,2826.38,39,169


In [9]:
feat_1 = data.groupby(['customer_id','coupon_id'])['quantity'].sum().reset_index()
feat_1 = feat_1.rename(columns={'quantity':'sum_trx_cust_cat_qty'})
feat_2 = data.groupby(['customer_id','coupon_id'])['selling_price'].sum().reset_index()
feat_2 = feat_2.rename(columns={'selling_price':'sum_trx_cust_cat_price'})

In [10]:
driver = driver.merge(feat_1, on=['customer_id','coupon_id'], how='left')
driver = driver.merge(feat_2, on=['customer_id','coupon_id'], how='left')
driver = driver.fillna(0)

In [11]:
driver = driver.drop(['campaign_id','coupon_id','customer_id'], axis=1)

In [12]:
driver.to_csv('../data/feature/tranx_category_feature.csv', index=False)

In [13]:
driver.shape

(128595, 3)

In [14]:
driver.head()

,id,sum_trx_cust_cat_qty,sum_trx_cust_cat_price
0,1,328.0,22561.95
1,2,361.0,22781.08
2,6,95.0,12398.52
3,7,232.0,15645.29
4,9,722.0,34023.81
